# Lesson 05 · Bayesian A/B testing

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 6 (pp. 45–54)

**Question:** Mike Piazza hit .308 over his career and Hank Aaron hit .305. Is Piazza really the better hitter, or did he just get luckier?

**Goal:** Compute the probability that one player's true average beats another's in four ways: simulation, numerical integration, an exact formula and a normal approximation. Then build a credible interval for the difference and compare it with the classical test.

**Contents**

1. Setup (a new pitcher rule)
2. Two posteriors side by side
3. Route 1: simulation
4. Route 2: numerical integration
5. Route 3: the exact formula
6. Route 4: normal approximation
7. Credible vs confidence intervals for a difference
8. Exercises

**Tags:** `ab-testing` `comparing-proportions` `posterior-simulation` `numerical-integration` `normal-approximation` `chi-squared`

**Before you start:** Lesson 03.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

# From here on the book only drops players who pitched in MORE than 3 games
# (so Ty Cobb, who pitched a couple of times, stays in).
career = eb_data.career(pitcher_rule=3)
alpha0, beta0 = eb_data.load_prior()
print(f"{len(career):,} players | prior alpha0 = {alpha0:.1f}, beta0 = {beta0:.1f}")

## 1. Setup

**Your turn.** Add `eb_estimate`, `alpha1` and `beta1`. Pull out the rows for Hank Aaron, Mike Piazza and Hideki Matsui.

In [ ]:
# Your code here

## 2. Two posteriors side by side

**Your turn.** Plot the posterior densities of Aaron and Piazza, then add Matsui.

In [ ]:
# Your code here

**Reflect.** From the picture alone, roughly what's the probability that Piazza is better than Aaron? What about Matsui versus Piazza?

*Your answer:*

## 3. Route 1: simulation

Draw many values from each posterior and count how often Piazza's draw is bigger. No maths needed, and it extends to almost any model. This is the same reason MCMC works.

**Your turn.** Estimate P(p_Piazza > p_Aaron) with 1,000,000 draws each. Time it with `%timeit` (or `time.perf_counter`).

In [ ]:
# Your code here

> **Check:** ≈ 0.595.

## 4. Route 2: numerical integration

The two posteriors are independent, so their joint density is the product. We want the mass of the region where $x_{Piazza} > y_{Aaron}$:

$$\Pr = \iint_{x > y} f_P(x)\, f_A(y)\, dx\, dy.$$

**Your turn.** Approximate the integral on a grid from 0.29 to 0.33 with spacing 0.00002 (or coarser at first). `np.meshgrid` or broadcasting with `[:, None]` will help. Plot the joint density as a contour with the line x = y.

In [ ]:
# Your code here

In [ ]:
# Your code here

## 5. Route 3: the exact formula

Evan Miller derived a closed form for two beta posteriors. With $A \sim \text{Beta}(\alpha_A, \beta_A)$ and $B \sim \text{Beta}(\alpha_B, \beta_B)$, and $\alpha_B$ an integer:

$$\Pr(p_B > p_A) = \sum_{i=0}^{\alpha_B - 1} \frac{B(\alpha_A + i,\; \beta_A + \beta_B)}{(\beta_B + i)\, B(1 + i,\; \beta_B)\, B(\alpha_A, \beta_A)},$$

where $B$ is the beta function. Work on the log scale with `special.betaln` to avoid overflow. Our $\alpha$ values aren't whole numbers, so round $\alpha_B$ as the book does.

**Your turn.** Write `prob_b_beats_a(alpha_a, beta_a, alpha_b, beta_b)` and compute P(Piazza > Aaron), with Piazza as B.

In [ ]:
# Your code here

> **Check:** ≈ 0.59–0.60. Rounding α_B shifts the last digit, so the routes agree to about two decimal places.

## 6. Route 4: normal approximation

With large $\alpha$ and $\beta$, a beta looks normal, with mean $\alpha/(\alpha+\beta)$ and variance $\alpha\beta / ((\alpha+\beta)^2(\alpha+\beta+1))$. The difference of two independent normals is normal, so the probability is one call to `stats.norm.cdf`.

**Your turn.** Write `prob_b_beats_a_approx(...)`. It should work on whole arrays. Compare all four routes in a table with their run times.

In [ ]:
# Your code here

**Reflect.** When would the normal approximation go wrong? Why is it safe with a prior as strong as ours?

*Your answer:*

## 7. Credible vs confidence intervals for a difference

**Your turn.** Classical: build the 2×2 table of hits and misses for Aaron and Piazza and run `stats.chi2_contingency`. Then compute a 95% Wald confidence interval for the difference in proportions.

In [ ]:
# Your code here

> **Check:** p-value ≈ 0.7; 95% CI for the difference ≈ (−0.0165, 0.0109).

**Your turn.** Bayesian: use the normal approximation to get a 95% credible interval for p_Piazza − p_Aaron, along with the posterior probability.

In [ ]:
# Your code here

**Your turn.** Pick 20 random players and compare each with Piazza: plot each player's credible interval and confidence interval for the difference, sorted by AB.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Same question as Lesson 03: when do the two kinds of interval agree, and when does the prior make a difference?

*Your answer:*

## 8. Exercises

**Your turn.** Use your vectorized normal approximation to compute, for every player in history, the probability that he's a better hitter than Piazza. How many exceed 95%? Who are they?

In [ ]:
# Your code here

**Your turn.** **Card connection.** Two eBay sellers list the same graded card. Seller A sold 18 of 40 listings and Seller B sold 7 of 12. Using a beta prior that you justify, what's the probability that A's true sell-through rate is higher? Try at least two routes from this lesson.

In [ ]:
# Your code here

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 6.
- Evan Miller, "Formulas for Bayesian A/B testing": https://www.evanmiller.org/bayesian-ab-testing.html
- Chris Stucchio, "Bayesian A/B testing at VWO" (whitepaper), for the normal approximation.